<a href="https://colab.research.google.com/github/VukasinA/ML_projekti/blob/main/drugi_projekat.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pycuda

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 85.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.8/98.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.2/103.2 kB 11.6 MB/s eta 0:00:00
  Created wheel for pycuda: filename=pycuda-2025.1.2-cp312-cp312-linux_x86_64.whl size=659050 sha256=6db70d79b187332eb5fc1a4f84669cab785e1bb9148c3399a4cc29a6fde263ec
  Stored in directory: /root/.cache/pip/wheels/d5/36/f3/ac5f09d768cad3fa15d5a3449bdfe65c3de58e69d036c73228
Successfully built pycuda


In [ ]:
!nvidia-smi

Sun Jan 11 22:38:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
import pycuda.autoinit
import pycuda.driver as cuda
from pycuda.compiler import SourceModule
import numpy as np
import time

# kerneli
cuda_prvi = r"""
__global__ void diffusion_step_kernel(
    float *conc_in,
    float *conc_out,
    int H, int W,
    float decay)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= W || y >= H) return;

    int idx = y * W + x;

    float self = conc_in[idx];
    float up    = (y > 0)   ? conc_in[(y-1)*W + x] : 0.0f;
    float down  = (y < H-1) ? conc_in[(y+1)*W + x] : 0.0f;
    float left  = (x > 0)   ? conc_in[y*W + (x-1)] : 0.0f;
    float right = (x < W-1) ? conc_in[y*W + (x+1)] : 0.0f;

    float avg = (self + up + down + left + right) / 5.0f;
    avg = avg * (1.0f - decay);

    conc_out[idx] = avg;
}

__global__ void apply_sources_kernel(
    float *conc,
    unsigned char *cell_type,
    int H, int W,
    float source_value,
    float absorb_value)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= W || y >= H) return;

    int idx = y * W + x;

    if (cell_type[idx] == 1)
        conc[idx] = source_value;
    else if (cell_type[idx] == 2) {
        float v = conc[idx] - absorb_value;
        conc[idx] = (v > 0.0f) ? v : 0.0f;
    }
}
__global__ void clamp_kernel(
    float *conc,
    int H, int W,
    float C_max)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= W || y >= H) return;

    int idx = y * W + x;

    float v = conc[idx];
    if (v < 0.0f) v = 0.0f;
    if (v > C_max) v = C_max;

    conc[idx] = v;
}
"""

cuda_drugi = r"""
__global__ void count_danger_kernel(
    const float *conc,
    int *danger_count,
    int H, int W,
    float danger_threshold)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= W || y >= H) return;

    int idx = y * W + x;

    if (conc[idx] >= danger_threshold) {
        danger_count[idx] += 1;
    }
}

__global__ void count_ever_dangerous_kernel(
    const int *danger_count,
    int H, int W,
    int *danger_counter)
{
    int x = blockIdx.x * blockDim.x + threadIdx.x;
    int y = blockIdx.y * blockDim.y + threadIdx.y;

    if (x >= W || y >= H) return;

    int idx = y * W + x;

    if (danger_count[idx] > 0) {
        atomicAdd(danger_counter, 1);
    }
}
"""

H, W = 1000, 1000
T = 100

decay = np.float32(0.01)
source_value = np.float32(1.0)
absorb_value = np.float32(0.05)

danger_threshold = np.float32(0.2)   # prag

# podaci
conc = np.zeros((H, W), dtype=np.float32)
cell_type = np.zeros((H, W), dtype=np.uint8)

# izvor u centru
cell_type[H // 2, W // 2] = 1

# cistacka zona
cell_type[H // 3:H // 3 + 20, W // 3:W // 3 + 20] = 2

danger_count = np.zeros((H, W), dtype=np.int32)

# gpu alloc
conc_old_gpu = cuda.mem_alloc(conc.nbytes)
conc_new_gpu = cuda.mem_alloc(conc.nbytes)
cell_type_gpu = cuda.mem_alloc(cell_type.nbytes)
danger_count_gpu = cuda.mem_alloc(danger_count.nbytes)

cuda.memcpy_htod(conc_old_gpu, conc)
cuda.memcpy_htod(cell_type_gpu, cell_type)
cuda.memset_d32(danger_count_gpu, 0, H * W)

# kerneli
mod = SourceModule(cuda_prvi + cuda_drugi)

#dodamo ovde c_max koji ce da bude i kasnije ubacimo clamp_kernel dole gde i ostale

diffusion_kernel = mod.get_function("diffusion_step_kernel")
apply_kernel = mod.get_function("apply_sources_kernel")
count_danger_kernel = mod.get_function("count_danger_kernel")
count_ever_kernel = mod.get_function("count_ever_dangerous_kernel")

#256 niti po bloku
block = (16, 16, 1)
grid = ((W + 15) // 16, (H + 15) // 16)

# simulacija
start = time.time()

for _ in range(T):
    diffusion_kernel(
        conc_old_gpu,
        conc_new_gpu,
        np.int32(H),
        np.int32(W),
        decay,
        block=block,
        grid=grid
    )

    apply_kernel(
        conc_new_gpu,
        cell_type_gpu,
        np.int32(H),
        np.int32(W),
        source_value,
        absorb_value,
        block=block,
        grid=grid
    )

    # 2.1
    count_danger_kernel(
        conc_new_gpu,
        danger_count_gpu,
        np.int32(H),
        np.int32(W),
        danger_threshold,
        block=block,
        grid=grid
    )

    # ping pong
    conc_old_gpu, conc_new_gpu = conc_new_gpu, conc_old_gpu

cuda.Context.synchronize()
end = time.time()

print(f"vreme simulacije {H}x{W} T={T}: {end - start:.3f} s")

# 2.2
danger_counter_host = np.zeros(1, dtype=np.int32)
danger_counter_gpu = cuda.mem_alloc(danger_counter_host.nbytes)
cuda.memcpy_htod(danger_counter_gpu, danger_counter_host)

count_ever_kernel(
    danger_count_gpu,
    np.int32(H),
    np.int32(W),
    danger_counter_gpu,
    block=block,
    grid=grid
)

cuda.Context.synchronize()
cuda.memcpy_dtoh(danger_counter_host, danger_counter_gpu)

print("broj opasnih celija:", int(danger_counter_host[0]))

vreme simulacije 1000x1000 T=100: 0.015 s
broj opasnih celija: 25
